In [34]:
import pandas as pd
import numpy as np
import datetime

from pathlib import Path 

In [35]:
current_dir = Path(Path.cwd()).parent
data_dir = current_dir / "data"
source_data_dir = data_dir / "source"
print(data_dir)

c:\Users\pedro\DEV\dengue_prediction\dengue_prediction\data


In [36]:

import re
import unicodedata
 
def show_columns(df, n_range = 7):
    df = list(df.columns)
    l = []
    range = n_range
    for c in df:
        if range == n_range:
            print(l)
            l = []
            range = 0
        range += 1
        l.append(c)
    print("\n")

def normalize_column_name(col):
    # tira acentos
    col = unicodedata.normalize("NFKD", col).encode("ascii", "ignore").decode("utf-8")
    # minúsculas
    col = col.lower()
    # troca qualquer coisa que não seja letra ou número por _
    col = re.sub(r"[^a-z0-9]+", "_", col)
    # remove _ no começo e no fim
    col = col.strip("_")
    return col

# DEGBR

In [37]:

def prep_DEGBR(df:pd.DataFrame, df_cod_municip:pd.DataFrame):
    # id municipio para municipio e UF
    # id_municip -> códigos IBGE
    df = df.merge(
        df_cod_municip,
        left_on="ID_MUNICIP",
        right_on="codigo_municipio",
        how="left"
    )
    # filtra colunas e renomea-as
    df = df[["DT_NOTIFIC", "uf", "nome"]].rename(columns={
        "DT_NOTIFIC": "data",
        "nome": "municipio"
    })
    # garantir o formato
    df["data"] = pd.to_datetime(df["data"], errors="coerce").dt.normalize()
    # Remove colunas com valores ausentes
    df = df.dropna()
    # agrupar
    df = df.groupby(list(df.columns)).size().reset_index(name="casos_dengue")
    # Normalizacao nome colunas 
    df.columns = [normalize_column_name(col) for col in df.columns]
    # Normalizaco colunas nao numericas
    df["uf"] = df["uf"].astype(str).str.strip().str.upper()
    df["municipio"] = df["municipio"].astype(str).str.strip().str.lower()   
    return df 

def get_municip_map_df(cod_munipc_uniques):
    # load csv
    municipios = pd.read_csv(
        source_data_dir / "municipios.csv",
        usecols=["codigo_ibge", "nome", "codigo_uf"],
        sep=","
    )

    # load csv
    estados = pd.read_csv(
        source_data_dir / "estados.csv",
        usecols=["codigo_uf", "uf"],
        sep=","
    )
    
    municipios = municipios.merge(
        estados,
        on="codigo_uf",
        how="left"
    )
    # Remove coluna
    municipios = municipios.drop(columns=["codigo_uf"])
    
    # Ajusta e filtra
    municipios["codigo_municipio"] = municipios["codigo_ibge"] // 10
    municipios = municipios.drop(columns=["codigo_ibge"])
    municipios_filtrados = municipios[
        municipios["codigo_municipio"].isin(cod_munipc_uniques)
    ].copy()
    
    return municipios_filtrados

In [38]:
dengBr_dir = source_data_dir / "DENGBR" 
dengBr23_file = dengBr_dir / "DENGBR23.csv"
dengBr24_file = dengBr_dir / "DENGBR24.csv"
dengBr25_file = dengBr_dir / "DENGBR25.csv"

dengBr23 = pd.read_csv(dengBr23_file, usecols=["DT_NOTIFIC", "ID_MUNICIP", "ID_AGRAVO"], sep=",")
dengBr24 = pd.read_csv(dengBr24_file, usecols=["DT_NOTIFIC", "ID_MUNICIP", "ID_AGRAVO"], sep=",")
dengBr25 = pd.read_csv(dengBr25_file, usecols=["DT_NOTIFIC", "ID_MUNICIP", "ID_AGRAVO"], sep=",")

cod_munipc_uniques = dengBr23["ID_MUNICIP"].unique().tolist()
cod_munipc_uniques.extend(dengBr24["ID_MUNICIP"].unique().tolist())
cod_munipc_uniques.extend(dengBr25["ID_MUNICIP"].unique().tolist())

df_cod_municip = get_municip_map_df(cod_munipc_uniques)

dengBr23 = prep_DEGBR(dengBr23, df_cod_municip)
dengBr24 = prep_DEGBR(dengBr24, df_cod_municip)
dengBr25 = prep_DEGBR(dengBr25, df_cod_municip)

df_DEGBR = pd.concat([dengBr23, dengBr24, dengBr25], ignore_index=True)

# INMET

In [39]:
def prep_INMET(df: pd.DataFrame, file_name):
    # Remove coluna vazia gerada por ; no final da linha
    df = df.loc[:, ~df.columns.str.contains("^Unnamed")].copy()

    # Remove coluna com muitos valores ausentes
    df = df.drop(
        columns=[
            col for col in df.columns
            if "RADIACAO GLOBAL" in col
        ],
        errors="ignore"
    )

    # Converte a coluna de data
    df["Data"] = pd.to_datetime(
        df["Data"],
        format="%Y/%m/%d",
        errors="coerce"
    )

    # Colunas numéricas
    cols_nao_numericas = ["Data", "Hora UTC"]

    num_cols = [
        col for col in df.columns
        if col not in cols_nao_numericas
    ]

    # Converte colunas numéricas
    for col in num_cols:
        df[col] = (
            df[col]
            .astype(str)
            .str.strip()
            .str.replace(",", ".", regex=False)
        )

        df[col] = pd.to_numeric(df[col], errors="coerce")

    # Preenche ausentes com média móvel
    df[num_cols] = df[num_cols].fillna(
        df[num_cols].rolling(window=4, min_periods=2).mean()
    )
    # Remove linhas com valroes ausentes
    df = df.dropna()
    # Agrega por dia
    df = (
        df
        .groupby("Data", as_index=False)[num_cols]
        .mean()
    )

    # Pega UF e município
    partes_nome = file_name.split("_")

    uf = partes_nome[2]
    municipio = partes_nome[4]

    df["uf"] = uf.upper()
    df["municipio"] = municipio.lower()

    # Renomeia colunas
    df.columns = [normalize_column_name(col) for col in df.columns]

    return df
    

In [ ]:
dfs_diarios = []
inmet_dir = source_data_dir / "INMET"

for year in [2023, 2024, 2025]:
    year_dir = inmet_dir / str(year)

    for path in year_dir.iterdir():
        file_name = path.stem
        df = pd.read_csv(
            path,
            encoding="latin1",
            sep=";",
            skiprows=8
        )
        df = prep_INMET(df, file_name=file_name)
        dfs_diarios.append(df)
        
df_INMET = pd.concat(dfs_diarios, ignore_index=True)
# Garantir ordem
df_INMET = df_INMET.sort_values(["uf", "municipio", "data"])

## media dos 30 dias anteriores

In [ ]:
id_cols = ["uf", "municipio", "data"]

days_mean = df_INMET.groupby(id_cols, as_index=False).mean(numeric_only=True)

num_cols = [
    col for col in days_mean.columns
    if col not in id_cols
]

df_INMET_mean_30 = days_mean[id_cols].copy()

for col in num_cols:
    # shift(1) to dont get the current day, and rolling(30) to get the mean of the last 30 days
    df_INMET_mean_30[col] = (
        days_mean
        .groupby(["uf", "municipio"])[col]
        .transform(lambda x: x.shift(1).rolling(window=30, min_periods=30).mean())
    )

,uf,municipio,data,precipitacao_total_horario_mm,pressao_atmosferica_ao_nivel_da_estacao_horaria_mb,pressao_atmosferica_max_na_hora_ant_aut_mb,pressao_atmosferica_min_na_hora_ant_aut_mb,temperatura_do_ar_bulbo_seco_horaria_c,temperatura_do_ponto_de_orvalho_c,temperatura_maxima_na_hora_ant_aut_c,temperatura_minima_na_hora_ant_aut_c,temperatura_orvalho_max_na_hora_ant_aut_c,temperatura_orvalho_min_na_hora_ant_aut_c,umidade_rel_max_na_hora_ant_aut,umidade_rel_min_na_hora_ant_aut,umidade_relativa_do_ar_horaria,vento_direcao_horaria_gr_gr,vento_rajada_maxima_m_s,vento_velocidade_horaria_m_s
0,AC,epitaciolandia,2023-12-09,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,AC,epitaciolandia,2023-12-10,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,AC,epitaciolandia,2023-12-11,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,AC,epitaciolandia,2023-12-12,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,AC,epitaciolandia,2023-12-13,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
394995,TO,santa rosa do tocantins,2025-12-27,0.227039,976.360625,976.643572,976.055359,25.186250,21.548058,25.677360,24.658823,22.012810,21.074578,84.582613,78.923247,81.719090,170.058108,4.046397,1.899120
394996,TO,santa rosa do tocantins,2025-12-28,0.220372,976.381106,976.660630,976.076994,25.178056,21.566060,25.673257,24.642392,22.034413,21.093243,84.734643,79.034999,81.836718,171.965016,3.946934,1.866399
394997,TO,santa rosa do tocantins,2025-12-29,0.220372,976.347791,976.619658,976.036735,25.186889,21.568652,25.694424,24.638197,22.044876,21.092613,84.773995,78.982684,81.804403,171.166128,3.936759,1.851176
394998,TO,santa rosa do tocantins,2025-12-30,0.227908,976.383656,976.655747,976.067709,25.223278,21.583477,25.745356,24.657122,22.068068,21.095828,84.751612,78.852088,81.723807,174.125679,3.940501,1.831872


## serie temporal dos 30 dias anteriores

In [ ]:
df_INMET_last30 = df_INMET[id_cols].copy()

for col in num_cols:
    for i in range(1, 31):
        df_INMET_last30[f"{col}_dia_{i}"] = df_INMET.groupby(["uf", "municipio"])[col].shift(i)

df_INMET_last30

C:\Users\pedro\AppData\Local\Temp\ipykernel_53920\3122195335.py:5: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_INMET_last30[f"{col}_dia_{i}"] = df_INMET.groupby(["uf", "municipio"])[col].shift(i)
C:\Users\pedro\AppData\Local\Temp\ipykernel_53920\3122195335.py:5: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_INMET_last30[f"{col}_dia_{i}"] = df_INMET.groupby(["uf", "municipio"])[col].shift(i)
C:\Users\pedro\AppData\Local\Temp\ipykernel_53920\3122195335.py:5: PerformanceWarning: DataFrame is highly fragmented.  This is usua

KeyboardInterrupt: 

# CREAT DATASET (MERGE)

### Analise de casos por municipio

In [ ]:
# Linhas em que não encontrou informação de dengue
sem_dengue = dengBr23[dengBr23["casos_dengue"].isna()].copy()

# Municípios sem informação de dengue em pelo menos uma data
municipios_sem_dengue = (
    sem_dengue[["uf", "municipio"]]
    .drop_duplicates()
    .sort_values(["uf", "municipio"])
)

print(municipios_sem_dengue)
print("Qtd municípios sem informação:", municipios_sem_dengue.shape[0])

Empty DataFrame
Columns: [uf, municipio]
Index: []
Qtd municípios sem informação: 0


In [ ]:
exemplos_com_dengue_por_municipio = (
    dengBr23
    .dropna(subset=["casos_dengue"])
    .groupby(["uf", "municipio"])
    .size()
    .reset_index(name="qtd_exemplos_com_dengue")
    .sort_values("qtd_exemplos_com_dengue", ascending=False)
)

print(exemplos_com_dengue_por_municipio)

max_exemplos = exemplos_com_dengue_por_municipio["qtd_exemplos_com_dengue"].max()

bins = [
    0,
    max_exemplos * 0.25,
    max_exemplos * 0.50,
    max_exemplos * 0.75,
    max_exemplos + 1
]

labels = [
    "menos_de_1_quarto",
    "entre_1_e_2_quartos",
    "entre_2_e_3_quartos",
    "entre_3_e_4_quartos"
]

exemplos_com_dengue_por_municipio["faixa_qtd_exemplos"] = pd.cut(
    exemplos_com_dengue_por_municipio["qtd_exemplos_com_dengue"],
    bins=bins,
    labels=labels,
    right=False
)

resumo_faixas = (
    exemplos_com_dengue_por_municipio
    .groupby("faixa_qtd_exemplos", observed=False)
    .size()
    .reset_index(name="qtd_municipios")
)

print("Máximo de exemplos em um município:", max_exemplos)
print(resumo_faixas)

      uf                       municipio  qtd_exemplos_com_dengue
682   ES         cachoeiro de itapemirim                      396
744   ES                         vitória                      391
743   ES                      vila velha                      377
731   ES                           serra                      376
709   ES                        linhares                      370
...   ..                             ...                      ...
4536  TO                       combinado                        1
4587  TO       porto alegre do tocantins                        1
2365  PB  são sebastião de lagoa de roça                        1
3076  RJ                  rio das flores                        1
3079  RJ            santa maria madalena                        1

[4615 rows x 3 columns]
Máximo de exemplos em um município: 396
    faixa_qtd_exemplos  qtd_municipios
0    menos_de_1_quarto            3842
1  entre_1_e_2_quartos             558
2  entre_2_e_3_quartos    

## mean

In [ ]:
df = df_INMET_mean_30.merge(
    df_DEGBR,
    on=["data", "municipio", "uf"],
    how="left"
)

print(df.shape)

# Conta exemplos com dengue por município
exemplos_por_municipio = (
    df
    .dropna(subset=["casos_dengue"])
    .groupby(["uf", "municipio"])
    .size()
    .reset_index(name="qtd_exemplos_com_dengue")
)

# Calcula metade do máximo
max_exemplos = exemplos_por_municipio["qtd_exemplos_com_dengue"].max()
limite_minimo = max_exemplos / 2

# Municípios que têm mais da metade do máximo
municipios_validos = exemplos_por_municipio[
    exemplos_por_municipio["qtd_exemplos_com_dengue"] >= limite_minimo
][["uf", "municipio"]]

print("Máximo de exemplos:", max_exemplos)
print("Limite mínimo:", limite_minimo)
print("Qtd municípios mantidos:", municipios_validos.shape[0])

# Mantém só esses municípios
df = df.merge(
    municipios_validos,
    on=["uf", "municipio"],
    how="inner"
)

# Remove as linhas sem target e os primeiros 30 dias para cada municipio (ja que pefamos a medias dos 30 dias anterires e so temos essa informacoa a partir de 30 dias)
df = df.dropna()

# Data para numérico
df["ano"] = df["data"].dt.year
df["mes"] = df["data"].dt.month
df["dia"] = df["data"].dt.day
df["dia_da_semana"] = df["data"].dt.dayofweek

df = df.drop(columns=["data"])
df = df.reset_index(drop=True)
df.to_csv(data_dir/"processed/datasets/casos_de_dengue_dataset.csv", index=False)
print(df)

(395041, 4)
Máximo de exemplos: 1105
Limite mínimo: 552.5
Qtd municípios mantidos: 28
       uf            municipio  casos_dengue   ano  mes  dia  dia_da_semana
0      AC           rio branco           1.0  2023    1    5              3
1      AC           rio branco           1.0  2023    1    6              4
2      AC           rio branco           1.0  2023    1    8              6
3      AC           rio branco           5.0  2023    1    9              0
4      AC           rio branco           3.0  2023    1   10              1
...    ..                  ...           ...   ...  ...  ...            ...
20860  SP  presidente prudente          10.0  2025   12   27              5
20861  SP  presidente prudente          10.0  2025   12   28              6
20862  SP  presidente prudente          18.0  2025   12   29              0
20863  SP  presidente prudente          15.0  2025   12   30              1
20864  SP  presidente prudente           6.0  2025   12   31              2

[

In [ ]:
df_INMET_last30["data"]

58052    2023-12-09
58053    2023-12-10
58054    2023-12-11
58055    2023-12-12
58056    2023-12-13
            ...    
332780   2025-12-27
332781   2025-12-28
332782   2025-12-29
332783   2025-12-30
332784   2025-12-31
Name: data, Length: 395000, dtype: datetime64[ns]

## time series 

In [ ]:
df = df_INMET_last30.merge(
    df_DEGBR,
    on=["data", "municipio", "uf"],
    how="left"
)

print(df.shape)

# Conta exemplos com dengue por município
exemplos_por_municipio = (
    df
    .dropna(subset=["casos_dengue"])
    .groupby(["uf", "municipio"])
    .size()
    .reset_index(name="qtd_exemplos_com_dengue")
)

# Calcula metade do máximo
max_exemplos = exemplos_por_municipio["qtd_exemplos_com_dengue"].max()
limite_minimo = max_exemplos / 2

# Municípios que têm mais da metade do máximo
municipios_validos = exemplos_por_municipio[
    exemplos_por_municipio["qtd_exemplos_com_dengue"] >= limite_minimo
][["uf", "municipio"]]

print("Máximo de exemplos:", max_exemplos)
print("Limite mínimo:", limite_minimo)
print("Qtd municípios mantidos:", municipios_validos.shape[0])

# Mantém só esses municípios
df = df.merge(
    municipios_validos,
    on=["uf", "municipio"],
    how="inner"
)

# Remove as linhas sem target e os primeiros 30 dias para cada municipio (ja que pefamos a medias dos 30 dias anterires e so temos essa informacoa a partir de 30 dias)
df = df.dropna()

# Data para numérico
df["ano"] = df["data"].dt.year
df["mes"] = df["data"].dt.month
df["dia"] = df["data"].dt.day
df["dia_da_semana"] = df["data"].dt.dayofweek

df = df.drop(columns=["data"])
df = df.reset_index(drop=True)
df.to_csv(data_dir/"processed/datasets/casos_de_dengue_dataset_time_series.csv", index=False)
print(df)

(395041, 4)
Máximo de exemplos: 1105
Limite mínimo: 552.5
Qtd municípios mantidos: 28
       uf            municipio  casos_dengue   ano  mes  dia  dia_da_semana
0      AC           rio branco           1.0  2023    1    5              3
1      AC           rio branco           1.0  2023    1    6              4
2      AC           rio branco           1.0  2023    1    8              6
3      AC           rio branco           5.0  2023    1    9              0
4      AC           rio branco           3.0  2023    1   10              1
...    ..                  ...           ...   ...  ...  ...            ...
20860  SP  presidente prudente          10.0  2025   12   27              5
20861  SP  presidente prudente          10.0  2025   12   28              6
20862  SP  presidente prudente          18.0  2025   12   29              0
20863  SP  presidente prudente          15.0  2025   12   30              1
20864  SP  presidente prudente           6.0  2025   12   31              2

[